In [4]:
code = 'OVERNIGHT_MTM'
pickle_path = 'P:/PGC Data/PICKLE/'
parameter_path = f'Parameter_{code}.csv'
meta_data_path = f"Parameter_{code}_MetaData.csv"
output_csv_path = f'{code}_output/'

from pgcbacktest.BtParameters import *
from pgcbacktest.BacktestOptions import *

def get_parameter_data(code, parameter_path):
    
    parameter = pd.read_csv(parameter_path)
    for col in parameter.columns:
        if ('time' in col) and ('and' not in col):
            globals()[f'{col}'] = pd.to_datetime(parameter[col].dropna().str.replace(' ', '').str[0:5], format='%H:%M').dt.time.to_list()
        else:
            globals()[f'{col}'] = parameter[col].dropna().to_list()
            
    parameter = pd.DataFrame(list(itertools.product(*[globals()[f'{col}'] for col in parameter.columns])), columns=parameter.columns)
    parameter['orderside'] = parameter['orderside'].str.upper()
    
    parameter.drop_duplicates(inplace=True, ignore_index=True)
    return parameter, len(parameter)
            
try:
    parameter, parameter_len = get_parameter_data(code, parameter_path)    
    meta_data, meta_row_nos = get_meta_data(code, meta_data_path)
    os.makedirs(output_csv_path, exist_ok=True)
except Exception as e:
    input(str(e))

In [5]:
def OVERNIGHT_MTM (bt:WeeklyBacktest, start_time, end_time, om, orderside):
    # try:
        start_dt = datetime.datetime.combine(bt.current_week_dates[0], start_time)        
        entry_time = start_dt
        if len(bt.current_week_dates) < 2:
            return
        re_straddle = []
        total_pnl =  0
        for i in range(maxre):
            start_dt = datetime.datetime.combine(start_dt.date(), start_time)
            
            try:    
                end_dt = datetime.datetime.combine(bt.current_week_dates[i+1], end_time)
            except:
                re_straddle+=['', '', '', '', '', '', '']
                continue
            
            ce_scrip, pe_scrip, ce_price, pe_price, future_price, start_dt = bt.get_strike(start_dt, end_dt, om=om)
            if ce_scrip is None:
                return None
            ce_unit, pe_unit = capital//ce_price, capital//pe_price
            ce_data = bt.get_single_leg_data(start_dt, end_dt, ce_scrip).copy()
            pe_data = bt.get_single_leg_data(start_dt, end_dt, pe_scrip).copy()
            
            ce_pnl = ce_data["close"].iloc[-1] - ce_price if orderside=="BUY" else ce_price - ce_data["close"].iloc[-1]
            pe_pnl = pe_data["close"].iloc[-1] - pe_price if orderside=="BUY" else pe_price - pe_data["close"].iloc[-1]

            ce_pnl -= bt.Cal_slipage(ce_price)
            pe_pnl -= bt.Cal_slipage(pe_price)
            pnl = ce_pnl*ce_unit + pe_pnl*pe_unit
            total_pnl+=pnl
            re_straddle += [start_dt, ce_price, pe_price, end_dt, ce_pnl, pe_pnl, pnl]

            start_dt = end_dt
        return [code, bt.index, start_time, end_time, orderside, om,bt.current_week_dates[0].date(), bt.current_week_dates[-1].date(), bt.from_dte, bt.to_dte, len(bt.current_week_dates), entry_time, future_price] + re_straddle + [total_pnl]
    # except Exception as e:
    #     print(e, [code, bt.index, start_time, end_time, orderside, om, bt.current_week_dates[0].date(), bt.current_week_dates[-1].date(), bt.from_dte, bt.to_dte, len(bt.current_week_dates), entry_time, future_price])
    #     return



In [ ]:
for row_idx in range(len(meta_data)):
    
    if row_idx in meta_row_nos and meta_data.loc[row_idx, 'run']:
        # try:
            meta_row = meta_data.iloc[row_idx]
            index, from_dte, to_dte, from_date, to_date, start_time, end_time, week_lists = get_meta_row_data(meta_row, pickle_path, weekly=True)
            dte_file = get_dte_file(pickle_path)
            maxre = 4
            log_cols = 'P_Strategy/P_Index/P_StartTime/P_EndTime/P_OrderSide/P_OM/Start.Date/End.Date/Start.DTE/End.DTE/DayCount/EntryTime/Future'
            capital = 10000
            for r in range(maxre):
                log_cols += f'/STD{r}.Time/STD{r}.CEPrice/STD{r}.PEPrice/STD{r}.EndTime/STD{r}.CEPnl/STD{r}.PEPnl/STD{r}.PNL' 
            log_cols += '/TotalPNL'
            log_cols = log_cols.split('/')
            
            for week_dates in week_lists:
                from_date = week_dates[0]
                to_date = week_dates[-1]

                file_name = f"{index} {week_dates[0].date()} {week_dates[-1].date()} {from_dte}-{to_dte} {code}"
                if not is_file_exists(output_csv_path, file_name, parameter_len):
                    t1 = datetime.datetime.now()
                    print(f"Row-{row_idx} | File-{file_name} | Total-{parameter_len}")

                    wbt = WeeklyBacktest(pickle_path, index, week_dates, from_dte, to_dte, start_time, end_time)
                    
                    
                    for idx, i in enumerate(range(0, parameter_len, chunk_size), start=1):
                        chunck_file_name = f"{output_csv_path}{file_name} No-{idx}.parquet"
                        print(chunck_file_name)

                        chunk_parameter = parameter.iloc[i:i+chunk_size]
                        chunk = [OVERNIGHT_MTM(wbt, row['entry_time'], row['exit_time'], row['om'], row['orderside']) for idx, row in tqdm(chunk_parameter.iterrows(), total=len(chunk_parameter), colour='GREEN')]
                        save_chunk_data(chunk, log_cols, chunck_file_name)
                        
                        del chunk
                        del chunk_parameter
                        gc.collect()
                    del wbt
                    gc.collect()
                    
                    t2 = datetime.datetime.now()
                    print(t2-t1)
        # except Exception as e:
        #     input(str(e))
            

Row-0 | File-NIFTY 2019-05-17 2019-05-23 5-1 OVERNIGHT_MTM | Total-108
OVERNIGHT_MTM_output/NIFTY 2019-05-17 2019-05-23 5-1 OVERNIGHT_MTM No-1.parquet


100%|██████████| 108/108 [00:03<00:00, 32.86it/s]


Deleting instance ...
0:00:03.793919
Row-0 | File-NIFTY 2019-05-24 2019-05-30 5-1 OVERNIGHT_MTM | Total-108
OVERNIGHT_MTM_output/NIFTY 2019-05-24 2019-05-30 5-1 OVERNIGHT_MTM No-1.parquet


100%|██████████| 108/108 [00:02<00:00, 40.92it/s]


Deleting instance ...
0:00:03.056865
Row-0 | File-NIFTY 2019-05-31 2019-06-06 5-1 OVERNIGHT_MTM | Total-108
OVERNIGHT_MTM_output/NIFTY 2019-05-31 2019-06-06 5-1 OVERNIGHT_MTM No-1.parquet


...

100%|██████████| 108/108 [00:02<00:00, 42.18it/s]


Deleting instance ...
0:00:02.984762
Row-0 | File-NIFTY 2021-03-12 2021-03-18 5-1 OVERNIGHT_MTM | Total-108
OVERNIGHT_MTM_output/NIFTY 2021-03-12 2021-03-18 5-1 OVERNIGHT_MTM No-1.parquet


100%|██████████| 108/108 [00:04<00:00, 23.15it/s]


Deleting instance ...
0:00:05.191408
Row-0 | File-NIFTY 2021-03-19 2021-03-25 5-1 OVERNIGHT_MTM | Total-108
OVERNIGHT_MTM_output/NIFTY 2021-03-19 2021-03-25 5-1 OVERNIGHT_MTM No-1.parquet


  0%|          | 0/108 [00:00<?, ?it/s]